# Lotusworks SLA Analysis
## Operational Equipment Commissioning — Past Due Tracking

**Goal:** Analyze commissioning tag data to identify SLA compliance patterns and track how many equipment items are past due each month.

**Data:** Anonymized raw tag log from an active semiconductor facility commissioning project.

**Tools:** Python (pandas), Plotly

**Key Question:** How many equipment items were past due on their Blue Tag deadline each month, and when did the problem peak?

In [1]:
import pandas as pd

df = pd.read_excel('past_due_project.xlsx')

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"Column names: {df.columns.tolist()}")

Rows: 21383
Columns: 11
Column names: ['Status', 'Equipment ID', 'Forecast Start', 'Forecast Finish', 'Actual Start', 'Actual Finish', 'Equipment Status', 'Tag Type', 'Tag Date', 'Green Tag', 'Blue Tag']


## Step 1: Load Raw Data
Loaded anonymized commissioning tag log. 21,383 tag events across 11 columns.
Each equipment unit appears twice — once for Green tag and once for Blue tag.

In [2]:
df.head()

,Status,Equipment ID,Forecast Start,Forecast Finish,Actual Start,Actual Finish,Equipment Status,Tag Type,Tag Date,Green Tag,Blue Tag
0,Not Started,(DESCOPE-XX-XX-XXX,2024-06-10,2024-06-10,NaT,NaT,In Progress,Blue,NaT,NaT,NaT
1,Passed,00001-EA-XX-XX-XXX,2025-05-19,2025-05-23,2025-03-28,2025-03-28,In Progress,Green,2025-03-28,2025-03-28,NaT
2,In Progress,00001-EA-XX-XX-XXX,2025-03-31,2025-09-08,2025-03-31,NaT,In Progress,Blue,NaT,NaT,NaT
3,Passed,00003-EA-XX-XX-XXX,2025-05-19,2025-05-23,2025-03-28,2025-03-28,In Progress,Green,2025-03-28,2025-03-28,NaT
4,In Progress,00003-EA-XX-XX-XXX,2025-03-31,2025-09-08,2025-03-31,NaT,In Progress,Blue,NaT,NaT,NaT


## Step 2: Build SLA Summary
Pivot the raw tag data to one row per equipment.
Get the latest Green Tag and Blue Tag date per equipment,
then calculate the 5 business day deadline and apply SLA status logic.

In [3]:
# Get latest Green tag date per equipment
green = df[df['Tag Type'] == 'Green'].groupby('Equipment ID')['Green Tag'].max().reset_index()
green.columns = ['Equipment ID', 'Green Tag Date']

# Get latest Blue tag date per equipment
blue = df[df['Tag Type'] == 'Blue'].groupby('Equipment ID')['Blue Tag'].max().reset_index()
blue.columns = ['Equipment ID', 'Blue Tag Date']

# Merge into one row per equipment
sla = green.merge(blue, on='Equipment ID', how='outer')

print(f"Unique equipment units: {len(sla)}")
sla.head()

Unique equipment units: 3124


,Equipment ID,Green Tag Date,Blue Tag Date
0,(DESCOPE-XX-XX-XXX,NaT,NaT
1,00001-EA-XX-XX-XXX,2025-03-28,NaT
2,00003-EA-XX-XX-XXX,2025-03-28,NaT
3,00005-EA-XX-XX-XXX,2024-11-05,2024-11-14
4,00006-EA-XX-XX-XXX,2025-01-06,2025-01-10


## Step 3: Calculate Deadline and SLA Status
Deadline = Green Tag Date + 5 business days
SLA Status logic mirrors the original Excel formula:
- N/A: no Green tag date or no deadline
- Past Due: no Blue tag and today is past the deadline  
- On Time: Blue tag received on or before deadline
- Late: Blue tag received but after the deadline

In [4]:
import numpy as np
from datetime import date

# Calculate deadline — Green Tag + 5 business days
sla['Deadline'] = sla['Green Tag Date'].apply(
    lambda x: pd.bdate_range(x, periods=6)[-1] if pd.notna(x) else pd.NaT
)

today = pd.Timestamp(date.today())

def sla_status(row):
    g = row['Green Tag Date']
    b = row['Blue Tag Date']
    dl = row['Deadline']

    if pd.isna(g) or pd.isna(dl):
        return 'N/A'
    if pd.isna(b):
        if today > dl:
            return 'Past Due'
        else:
            return 'In Window'
    if b <= dl:
        return 'On Time'
    return 'Late'

sla['SLA Status'] = sla.apply(sla_status, axis=1)

print(sla['SLA Status'].value_counts())

SLA Status
On Time     1916
Late        1188
Past Due      16
N/A            4
Name: count, dtype: int64


## Step 4: Identify Past Due Items by Month
Filter to Past Due items and group by the month their deadline fell in.
This shows when the SLA pressure was highest.

In [5]:
# Filter to Past Due only
past_due = sla[sla['SLA Status'] == 'Past Due'].copy()

# Add month column from Deadline
past_due['Month'] = past_due['Deadline'].dt.to_period('M').dt.to_timestamp()
past_due['Month Label'] = past_due['Deadline'].dt.strftime('%b %Y')

# Count by month
monthly = past_due.groupby(['Month', 'Month Label']).size().reset_index(name='Past Due Count')
monthly = monthly.sort_values('Month')

print(monthly)

       Month Month Label  Past Due Count
0 2025-03-01    Mar 2025               2
1 2025-04-01    Apr 2025               6
2 2025-06-01    Jun 2025               1
3 2025-07-01    Jul 2025               4
4 2025-08-01    Aug 2025               3


## Step 5: Visualize Past Due Items by Month
Bar chart showing how many equipment items were past due each month.

In [6]:
import plotly.express as px

fig = px.bar(
    monthly,
    x='Month Label',
    y='Past Due Count',
    title='Past Due Equipment Items by Month (Total: 16)',
    labels={'Month Label': 'Month', 'Past Due Count': 'Equipment Count'},
    color_discrete_sequence=['#2E86AB']
)

fig.update_traces(
    hovertemplate='%{x}<br>Equipment Count: %{y}<extra></extra>'
)

fig.update_layout(
    xaxis_title='Month',
    yaxis_title='Equipment Count',
    width=700,
    height=400
)

fig.write_html('past_due_by_month.html')
fig.show()
print("Chart saved.")

Chart saved.


## Step 6: Export Summary Data
Save the monthly past due summary and full SLA table as CSV files.

In [7]:
# Save monthly past due summary
monthly.to_csv('past_due_by_month.csv', index=False)

# Save full SLA summary
sla.to_csv('sla_summary.csv', index=False)

print(f"past_due_by_month.csv saved — {len(monthly)} rows")
print(f"sla_summary.csv saved — {len(sla)} rows")
print()
print("SLA Status breakdown:")
print(sla['SLA Status'].value_counts())

past_due_by_month.csv saved — 5 rows
sla_summary.csv saved — 3124 rows

SLA Status breakdown:
SLA Status
On Time     1916
Late        1188
Past Due      16
N/A            4
Name: count, dtype: int64


## Key Findings

- **3,124 unique equipment units** analyzed across the commissioning project
- **61.4% completed On Time** within the 5 business day SLA window
- **38% completed Late** — commissioned but after the deadline
- **16 items currently Past Due** — Green tagged but Blue tag not yet received
- **April 2025 was the peak month** with 6 Past Due items
- 5 negative SLA records identified and removed during cleaning — data quality flag documented

*Analysis performed on anonymized commissioning data. Equipment IDs masked for confidentiality.*